# Ex3_EKF-PEM-adaptive
Use the same fixed EKF histories and learn a parameter vector for each window.

With `LOAD_PRETRAINED = True`, load the pretrained parameters and evaluate the model. Set it to `False` to train. View training history with `tensorboard --logdir runs`.


In [ ]:
from utils import configure_runtime
configure_runtime()
from datetime import datetime
import numpy as np
import torch
from torch.utils.tensorboard import SummaryWriter
import copy

from data import load_data
from models import make_vehicle
from estimation import fixed_histories
from identification import AdaptivePEM
from utils import ROOT, load_initialization, save_metrics, show_metrics, show_model


## 1. Configuration

In [ ]:
EXPERIMENT = 3
NAME = "Ex3_EKF-PEM-adaptive"
LOAD_PRETRAINED = True
SEED = 0
system = dict(nx=3, nu=2, nw=3, N=40, T=0.04,
              E=np.eye(3), C=np.array([[1., 0., 0.], [0., 0., 1.]]),
              integration="rk4", rk4_steps=2, m=790.0, l1=1.248, l2=1.7328)
REFERENCE_DIR = ROOT / "runs" / NAME
OUTPUT_DIR = ROOT / "runs" / "new" / (NAME + "_" + datetime.now().strftime("%Y%m%d-%H%M%S"))
np.random.seed(SEED)
torch.manual_seed(SEED)
BATCH_SIZE, MAX_EPOCHS = 32, 2000
LEARNING_RATE = 6e-4
LR_PATIENCE, EARLY_STOPPING_PATIENCE = 50, 300
MIN_LR = 1e-6

system["hidden_limit"] = 3.0


## 2. Data and fixed EKF histories

In [ ]:
checkpoint = torch.load(REFERENCE_DIR / "model_params_opt.pth", map_location="cpu", weights_only=False)
data = load_data(EXPERIMENT)
initialization = load_initialization("Ex3_EKF-PEM")
initialization["parameter_initial"] = checkpoint["ekf_physics_params"]
ekf_model = make_vehicle(system, initialization)
histories = fixed_histories(ekf_model, data)


## 3. Parameter encoder and bounded physical model

In [ ]:
# The saved configuration contains the authors' architecture and the shared parameter bounds.
config = copy.deepcopy(checkpoint["config"])
config["MODEL"]["OPTIMIZATION"]["LR"] = LEARNING_RATE
config["MODEL"]["OPTIMIZATION"]["BATCH_SIZE"] = BATCH_SIZE
scaler = None
if LOAD_PRETRAINED:
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    scaler.mean_, scaler.scale_ = checkpoint["scaler_mean"], checkpoint["scaler_scale"]
    scaler.var_, scaler.n_features_in_ = scaler.scale_**2, len(scaler.mean_)
problem = AdaptivePEM(data, histories, config, system["C"], EXPERIMENT == 3, BATCH_SIZE, SEED, scaler)
model = problem.model
if LOAD_PRETRAINED:
    model.load_state_dict(checkpoint["state_dict"])


## 4. Optimize terminal PEM

In [ ]:
if not LOAD_PRETRAINED:
    OUTPUT_DIR.mkdir(parents=True)
    problem.generator = torch.Generator().manual_seed(SEED)
    optimizer = problem.model.optimizer
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, factor=.3, patience=LR_PATIENCE, threshold=1e-4, min_lr=MIN_LR)
    best_val, stale = float("inf"), 0
    with SummaryWriter(OUTPUT_DIR) as writer:
        for epoch in range(MAX_EPOCHS + 1):
            if epoch:
                problem.train_epoch()
            metrics = {s: problem.evaluate(s) for s in ("train", "val")}
            if epoch:
                scheduler.step(metrics["val"]["pem"])
            if metrics["val"]["pem"] < best_val - 1e-12:
                best_val, stale = metrics["val"]["pem"], 0
                problem.save(
                    OUTPUT_DIR / "model_params_opt.pth",
                    epoch,
                    metrics,
                    initialization["parameter_initial"]
                )
            else:
                stale += int(epoch > 0)
            for split in ("train", "val"):
                writer.add_scalar("PEM/" + split, metrics[split]["pem"], epoch)
            if epoch % 10 == 0:
                print(epoch, {s: m["pem"] for s, m in metrics.items()})
            if stale >= EARLY_STOPPING_PATIENCE and optimizer.param_groups[0]["lr"] <= MIN_LR*1.01:
                break
    selected = torch.load(OUTPUT_DIR / "model_params_opt.pth", weights_only=False)
    problem.model.load_state_dict(selected["state_dict"])


## 5. Model and prediction errors

In [ ]:
directory = REFERENCE_DIR if LOAD_PRETRAINED else OUTPUT_DIR
checkpoint = torch.load(directory / "model_params_opt.pth", map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
metrics = {s: problem.evaluate(s) for s in ("train", "val")}
if not LOAD_PRETRAINED:
    save_metrics(directory, metrics["train"]["pem"], metrics["val"]["pem"], checkpoint["epoch"])
parameter_names = [next(iter(p)) for p in checkpoint["config"]["PARAMETERS"]]
parameter_mean = dict(zip(parameter_names, metrics["val"]["parameter_mean"]))
print("Physical parameters below are means over validation windows.")
show_model(directory, model, parameters=parameter_mean)
show_metrics(metrics["train"]["pem"], metrics["val"]["pem"])


Physical parameters below are means over validation windows.
Checkpoint: runs/Ex3_EKF-PEM-adaptive/model_params_opt.pth
DeepDynamicsModelIAC(
  (feed_forward): ModuleList(
    (0): GRU(7, 39, num_layers=3, batch_first=True)
    (1): Flatten(start_dim=1, end_dim=-1)
    (2): Linear(in_features=1521, out_features=188, bias=True)
    (3): Mish()
    (4): Linear(in_features=188, out_features=188, bias=True)
    (5): Mish()
    (6): GuardLayer(
      (guard_dense): Linear(in_features=188, out_features=17, bias=True)
      (guard_activation): Sigmoid()
    )
  )
  (loss_function): MSELoss()
)
Model parameters: 349217
Bf         12.678954
Cf         0.6855408
Df         2839.6062
Ef       -0.33264619
Br         10.885491
Cr        0.80655432
Dr         1929.8965
Er       -0.49996525
Cm1        3130.5886
Cm2       0.40206119
Cr0        1.2874287
Cr2        1.1390433
Iz          2446.603
Shf      0.002534085
Svf        29.821215
Shr   -0.00082672009
Svr        70.948685
Train PEM cost: 2.201880